# 06 — Phase 5: Position Analysis
**Football Players Stats 2024/25**

Comparing how different positions play — attacking, defending, progressive actions, aerial ability, and dribbling.

> Outfield players with 900+ minutes only.

## Setup

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import numpy as np

player_df = pd.read_csv('Dataset\player_clean.csv')

POSITION_COLORS = {'FW': '#e63946', 'MF': '#2a9d8f', 'DF': '#457b9d'}
POSITION_ORDER  = ['FW', 'MF', 'DF']

pos_df = player_df[
    (player_df['sufficient_minutes'] == True) &
    (player_df['primary_position'] != 'GK')
].copy()

print(f"Outfield players with sufficient minutes: {len(pos_df)}")

Outfield players with sufficient minutes: 1457


In [2]:
# 5.1 Average Key Stats by Position
avg_by_pos = pos_df.groupby('primary_position').agg(
    avg_goals=('goals', 'mean'), avg_assists=('assists', 'mean'),
    avg_shots=('shots', 'mean'), avg_key_passes=('key_passes', 'mean'),
    avg_tackles=('tackles', 'mean'), avg_interceptions=('interceptions', 'mean'),
).reset_index().round(2)

avg_melted = avg_by_pos.melt(id_vars='primary_position', var_name='metric', value_name='average')
metric_labels = {
    'avg_goals': 'Goals', 'avg_assists': 'Assists', 'avg_shots': 'Shots',
    'avg_key_passes': 'Key Passes', 'avg_tackles': 'Tackles', 'avg_interceptions': 'Interceptions',
}
avg_melted['metric'] = avg_melted['metric'].map(metric_labels)

fig = px.bar(avg_melted, x='metric', y='average', color='primary_position', barmode='group',
             color_discrete_map=POSITION_COLORS,
             title='Average Key Stats by Position (2024/25)',
             labels={'average': 'Season Average', 'metric': '', 'primary_position': 'Position'},
             category_orders={'primary_position': POSITION_ORDER}, text='average')
fig.update_traces(textposition='outside', texttemplate='%{text:.1f}')
fig.update_layout(height=500, plot_bgcolor='white',
                  yaxis=dict(showgrid=True, gridcolor='#f0f0f0'), legend_title='Position')
fig.show()

In [3]:
# 5.2 Touches in Opponent Box by Position
fig = px.box(pos_df, x='primary_position', y='touches_opp_box', color='primary_position',
             color_discrete_map=POSITION_COLORS,
             title='Touches in Opponent Box by Position (2024/25)',
             labels={'touches_opp_box': 'Touches in Opponent Box', 'primary_position': 'Position'},
             points='outliers', category_orders={'primary_position': POSITION_ORDER})
fig.update_layout(height=480, showlegend=False, plot_bgcolor='white',
                  yaxis=dict(showgrid=True, gridcolor='#f0f0f0'))
fig.show()

In [4]:
# 5.3 Progressive Actions by Position
prog_by_pos = pos_df.groupby('primary_position').agg(
    avg_prog_carries=('progressive_carries', 'mean'),
    avg_prog_passes=('progressive_passes', 'mean'),
    avg_prog_receptions=('progressive_receptions', 'mean'),
).reset_index().round(1)

prog_melted = prog_by_pos.melt(id_vars='primary_position', var_name='metric', value_name='average')
prog_labels = {
    'avg_prog_carries': 'Progressive Carries',
    'avg_prog_passes': 'Progressive Passes',
    'avg_prog_receptions': 'Progressive Receptions',
}
prog_melted['metric'] = prog_melted['metric'].map(prog_labels)

fig = px.bar(prog_melted, x='metric', y='average', color='primary_position', barmode='group',
             color_discrete_map=POSITION_COLORS,
             title='Average Progressive Actions by Position (2024/25)',
             labels={'average': 'Season Average', 'metric': '', 'primary_position': 'Position'},
             category_orders={'primary_position': POSITION_ORDER}, text='average')
fig.update_traces(textposition='outside', texttemplate='%{text:.0f}')
fig.update_layout(height=500, plot_bgcolor='white',
                  yaxis=dict(showgrid=True, gridcolor='#f0f0f0'), legend_title='Position')
fig.show()

In [5]:
# 5.4 Aerial Duel Win % by Position (min 20 aerial duels)
aerial_df = pos_df[(pos_df['aerial_duels_won'] + pos_df['aerial_duels_lost']) >= 20].copy()

fig = px.box(aerial_df, x='primary_position', y='aerial_duel_win_pct', color='primary_position',
             color_discrete_map=POSITION_COLORS,
             title='Aerial Duel Win % by Position (min 20 aerial duels)',
             labels={'aerial_duel_win_pct': 'Aerial Duel Win %', 'primary_position': 'Position'},
             points='outliers', category_orders={'primary_position': POSITION_ORDER})
fig.update_layout(height=480, showlegend=False, plot_bgcolor='white',
                  yaxis=dict(showgrid=True, gridcolor='#f0f0f0'))
fig.show()

In [6]:
# 5.5 Radar Chart — Position Profile
radar_stats  = ['goals', 'assists', 'shots', 'key_passes', 'tackles',
                'interceptions', 'progressive_carries', 'progressive_passes', 'touches_opp_box']
radar_labels = ['Goals', 'Assists', 'Shots', 'Key Passes', 'Tackles',
                'Interceptions', 'Prog Carries', 'Prog Passes', 'Box Touches']

radar_by_pos = pos_df.groupby('primary_position')[radar_stats].mean()
for col in radar_stats:
    col_min, col_max = radar_by_pos[col].min(), radar_by_pos[col].max()
    radar_by_pos[col] = ((radar_by_pos[col] - col_min) / (col_max - col_min) * 100).round(1)

fig = go.Figure()
for pos in POSITION_ORDER:
    values = radar_by_pos.loc[pos].tolist()
    values += values[:1]
    fig.add_trace(go.Scatterpolar(r=values, theta=radar_labels + [radar_labels[0]],
                                  fill='toself', name=pos,
                                  line_color=POSITION_COLORS[pos],
                                  fillcolor=POSITION_COLORS[pos], opacity=0.3))
fig.update_layout(polar=dict(radialaxis=dict(visible=True, range=[0, 100])),
                  title='Position Profile Radar — Normalized Averages (2024/25)',
                  height=550, showlegend=True, legend_title='Position')
fig.show()

In [7]:
# 5.6 Dribble Success Rate by Position (min 20 attempts)
dribble_df = pos_df[pos_df['dribbles_attempted'] >= 20].copy()

fig = px.box(dribble_df, x='primary_position', y='dribble_success_pct', color='primary_position',
             color_discrete_map=POSITION_COLORS,
             title='Dribble Success Rate by Position (min 20 attempts)',
             labels={'dribble_success_pct': 'Dribble Success %', 'primary_position': 'Position'},
             points='outliers', category_orders={'primary_position': POSITION_ORDER})
fig.update_layout(height=480, showlegend=False, plot_bgcolor='white',
                  yaxis=dict(showgrid=True, gridcolor='#f0f0f0'))
fig.show()